# SD 1.5 + ControlNet: Mask-Conditioned Diseased Leaf Generation

Fine-tunes a **ControlNet** branch on top of frozen **Stable Diffusion 1.5** to generate
GLS-diseased corn leaf images conditioned on a **binary full-leaf silhouette mask**.

## Comparison with custom DDPM (`diffusion_train_linux.ipynb`)
| Aspect | Custom DDPM | SD 1.5 + ControlNet |
|--------|-------------|---------------------|
| Backbone | Scratch UNet (~31M params) | SD 1.5 UNet (pretrained, 860M params frozen) |
| Noise space | Pixel (512×512×3) | Latent (64×64×4, 8× compressed) |
| Conditioning | FiLM + 4-ch concat | ControlNet hint encoder + cross-attn |
| Trainable params | All ~31M | ControlNet only ~360M |
| Warm-start | GAN-A Phase-1 checkpoint | ImageNet-scale SD 1.5 weights |
| T steps | 1000 (cosine) | 1000 (DDPM scheduler) |
| Inference | DDIM 50 steps | DDIM 50 steps |

## Architecture
```
Binary leaf mask (512×512, 1ch)
        ↓  ControlNet hint encoder (4×Conv-ReLU, trained from scratch)
        ↓  ControlNet encoder copy  (init from SD 1.5 UNet encoder, trainable)
SD 1.5 UNet decoder (frozen)  ←  residual features from ControlNet
        ↓
Predicted noise ε in latent space (64×64×4)
        ↓  VAE decode
Generated GLS diseased leaf (512×512, RGB)
```

Text prompt is fixed (`"grey leaf spot diseased corn leaf"`) — the model learns
visual conditioning from the mask geometry, not from text diversity.

## Prerequisites
Run **cell 2** once to install required packages, then restart kernel.

## Data
- **Diseased images**: `data/diseased/gls/` (263 images)
- **Leaf masks**: `data/masks/leaf/gls/` (generated by `diffusion_data_setup.ipynb`)

**Target hardware**: RTX 4060 8 GB (fp16, batch=1, grad accumulation=4, gradient checkpointing)

In [ ]:
# 1. Install required packages (run once, then restart kernel)
import subprocess, sys

pkgs = ['diffusers', 'transformers', 'accelerate']
missing = []
for pkg in pkgs:
    try:
        __import__(pkg)
    except ImportError:
        missing.append(pkg)

if missing:
    print(f'Installing: {missing}')
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '--quiet'] + missing)
    print('Done. Please restart the kernel, then re-run from cell 2.')
else:
    print('All packages already installed.')

In [ ]:
# 2. Imports
import os
import sys
import json
import time
import math
import random
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.amp import autocast, GradScaler
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as transforms
import torchvision.transforms.functional as TF
from PIL import Image
from tqdm import tqdm
import numpy as np
import matplotlib

if 'ipykernel' not in sys.modules:
    matplotlib.use('Agg')
import matplotlib.pyplot as plt

from diffusers import (
    AutoencoderKL,
    UNet2DConditionModel,
    DDPMScheduler,
    DDIMScheduler,
    StableDiffusionControlNetPipeline,
    ControlNetModel,
)
from transformers import CLIPTextModel, CLIPTokenizer

print('Imports OK.')
print(f'diffusers version: {__import__("diffusers").__version__}')
print(f'transformers version: {__import__("transformers").__version__}')

In [ ]:
# 3. Resolve repo root
from pathlib import Path

repo_root = Path.cwd().resolve()
if repo_root.name == 'gan':
    repo_root = repo_root.parents[1]
elif repo_root.name == 'src':
    repo_root = repo_root.parent
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))
print(f'Repo root: {repo_root}')

In [ ]:
# ====================================================================
# 4. CONFIGURATION
# ====================================================================

# ----- Pretrained model (downloaded once to ~/.cache/huggingface) -----
SD15_MODEL_ID = 'runwayml/stable-diffusion-v1-5'

# ----- Data paths -----
DISEASED_DIR  = os.path.join(str(repo_root), 'data', 'diseased', 'gls')
LEAF_MASK_DIR = os.path.join(str(repo_root), 'data', 'masks', 'leaf', 'gls')

# ----- Output directories -----
CHECKPOINT_DIR = os.path.join(str(repo_root), 'src', 'models', 'checkpoints')
LOG_DIR        = os.path.join(str(repo_root), 'results', 'logs')
SAMPLE_DIR     = os.path.join(str(repo_root), 'results', 'figures', 'sd15_controlnet_samples')
os.makedirs(CHECKPOINT_DIR, exist_ok=True)
os.makedirs(LOG_DIR,        exist_ok=True)
os.makedirs(SAMPLE_DIR,     exist_ok=True)

# ----- Resume training (set to None to start fresh) -----
RESUME_CHECKPOINT = None  # e.g. os.path.join(CHECKPOINT_DIR, 'controlnet_epoch50')

# ----- Text prompt (fixed — mask provides spatial conditioning) -----
PROMPT = 'grey leaf spot diseased corn leaf'

# ----- Training hyperparameters (tuned for RTX 4060 8GB) -----
IMAGE_SIZE      = 512
BATCH_SIZE      = 1      # VAE+UNet+ControlNet at fp16 fills ~7-8 GB
GRAD_ACCUM      = 4      # effective batch size = 4
NUM_EPOCHS      = 200
SAVE_EVERY      = 25
SAMPLE_EVERY    = 10
NUM_WORKERS     = 2
TRAIN_SPLIT     = 0.9    # 236 train / 27 test  (same split as custom DDPM)
SPLIT_SEED      = 42

# ----- Optimizer -----
LEARNING_RATE   = 1e-5   # low LR: fine-tuning atop pretrained SD 1.5 features
WEIGHT_DECAY    = 1e-2
LR_WARMUP_STEPS = 100

# ----- Augmentation (same as custom DDPM for fair comparison) -----
AUG_CROP_SCALE   = (0.80, 1.0)
AUG_COLOR_JITTER = dict(brightness=0.15, contrast=0.15, saturation=0.10, hue=0.03)

# ----- Inference -----
DDIM_STEPS          = 50
GUIDANCE_SCALE      = 7.5   # classifier-free guidance scale
CONTROLNET_SCALE    = 1.0   # how strongly mask conditioning is applied

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')
if device.type == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'GPU memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
print(f'Image size: {IMAGE_SIZE}x{IMAGE_SIZE}  Latent: 64x64x4')
print(f'Batch: {BATCH_SIZE} × grad_accum {GRAD_ACCUM} = effective {BATCH_SIZE * GRAD_ACCUM}')
print(f'Epochs: {NUM_EPOCHS}  |  Prompt: "{PROMPT}"')

In [ ]:
# 5. Dataset — paired (GLS diseased image, binary leaf mask)
#
# Same structure as LeafDiffusionDataset in diffusion_train_linux.ipynb:
#   - Image normalisation: [-1, 1]  (SD / DDPM standard)
#   - Mask: {0, 1} float, used as ControlNet hint
#   - Synchronized geometric augmentation on image + mask
#   - Color jitter on image only

class LeafControlNetDataset(Dataset):
    EXTS = ('.png', '.jpg', '.jpeg')

    def __init__(self, image_dir, mask_dir, image_size=512, split='train',
                 train_fraction=0.9, seed=42, augment=True):
        img_files  = sorted(f for f in os.listdir(image_dir)
                            if f.lower().endswith(self.EXTS))
        mask_files = sorted(f for f in os.listdir(mask_dir)
                            if f.lower().endswith(self.EXTS))

        assert len(img_files)  > 0, f'No images in {image_dir}'
        assert len(mask_files) > 0, f'No masks in {mask_dir}'
        assert len(img_files) == len(mask_files), (
            f'Image/mask count mismatch: {len(img_files)} vs {len(mask_files)}. '
            'Run diffusion_data_setup.ipynb first.')

        rng     = random.Random(seed)
        indices = list(range(len(img_files)))
        rng.shuffle(indices)
        n_train = int(len(indices) * train_fraction)
        idx     = indices[:n_train] if split == 'train' else indices[n_train:]

        self.image_files  = [img_files[i]  for i in idx]
        self.mask_files   = [mask_files[i] for i in idx]
        self.image_dir    = image_dir
        self.mask_dir     = mask_dir
        self.image_size   = image_size
        self.augment      = augment
        self.color_jitter = transforms.ColorJitter(**AUG_COLOR_JITTER)

    def __len__(self):
        return len(self.image_files)

    def __getitem__(self, index):
        img  = Image.open(os.path.join(self.image_dir,
                                       self.image_files[index])).convert('RGB')
        mask = Image.open(os.path.join(self.mask_dir,
                                       self.mask_files[index])).convert('L')

        img  = img.resize((self.image_size, self.image_size), Image.BILINEAR)
        mask = mask.resize((self.image_size, self.image_size), Image.NEAREST)

        if self.augment:
            do_hflip = random.random() < 0.5
            do_vflip = random.random() < 0.5
            angle    = random.choice([0, 90, 180, 270])
            i, j, h, w = transforms.RandomResizedCrop.get_params(
                img, scale=AUG_CROP_SCALE, ratio=(0.9, 1.1))

            for do, fn in [(do_hflip, TF.hflip), (do_vflip, TF.vflip)]:
                if do:
                    img  = fn(img)
                    mask = fn(mask)
            if angle:
                img  = TF.rotate(img, angle)
                mask = TF.rotate(mask, angle)

            img  = TF.resized_crop(img,  i, j, h, w,
                                   (self.image_size, self.image_size), Image.BILINEAR)
            mask = TF.resized_crop(mask, i, j, h, w,
                                   (self.image_size, self.image_size), Image.NEAREST)
            img  = self.color_jitter(img)

        img_t  = TF.to_tensor(img)   # [3, H, W] in [0, 1]
        mask_t = TF.to_tensor(mask)  # [1, H, W] in [0, 1]

        img_t  = img_t * 2.0 - 1.0                 # → [-1, 1]
        mask_t = (mask_t > 0.5).float()             # → {0, 1}

        # ControlNet hint: replicate mask to 3 channels (standard ControlNet format)
        hint_t = mask_t.repeat(3, 1, 1)             # [3, H, W] in {0, 1}

        return img_t, hint_t, self.image_files[index]


train_ds = LeafControlNetDataset(
    DISEASED_DIR, LEAF_MASK_DIR, IMAGE_SIZE,
    split='train', train_fraction=TRAIN_SPLIT, seed=SPLIT_SEED, augment=True)
test_ds  = LeafControlNetDataset(
    DISEASED_DIR, LEAF_MASK_DIR, IMAGE_SIZE,
    split='test',  train_fraction=TRAIN_SPLIT, seed=SPLIT_SEED, augment=False)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,
                          num_workers=NUM_WORKERS, pin_memory=True, drop_last=True)
test_loader  = DataLoader(test_ds,  batch_size=1, shuffle=False,
                          num_workers=NUM_WORKERS, pin_memory=True)

print(f'Train: {len(train_ds)} samples  |  Test: {len(test_ds)} samples')
print(f'Batches per epoch: {len(train_loader)}')
sample_img, sample_hint, _ = train_ds[0]
print(f'Image range:  [{sample_img.min():.3f}, {sample_img.max():.3f}]  (expected ~[-1, 1])')
print(f'Hint unique:  {sample_hint.unique().tolist()}  (expected [0.0, 1.0])')
print(f'Hint shape:   {tuple(sample_hint.shape)}  (3-channel for ControlNet)')

In [ ]:
# 6. Load SD 1.5 components + build ControlNet
#
# Trainable:   ControlNet weights only
# Frozen:      VAE, UNet, text encoder
#
# Memory budget (RTX 4060 8GB, fp16):
#   VAE         ~320 MB
#   UNet        ~1.7 GB
#   ControlNet  ~1.4 GB  (trainable)
#   Activations ~1-2 GB  (gradient checkpointing reduces this)
#   Optimizer   ~2.8 GB  (AdamW fp32 states for ControlNet)
#   Total       ~7.5 GB  ← fits on 8GB with care

print(f'Loading SD 1.5 from "{SD15_MODEL_ID}" ...')
print('  (First run downloads ~4 GB to ~/.cache/huggingface)')

# VAE — encode images to latents, decode latents to pixels
vae = AutoencoderKL.from_pretrained(
    SD15_MODEL_ID, subfolder='vae', torch_dtype=torch.float16).to(device)
vae.requires_grad_(False)
vae.eval()

# CLIP text encoder + tokenizer — encode the fixed prompt
tokenizer    = CLIPTokenizer.from_pretrained(SD15_MODEL_ID, subfolder='tokenizer')
text_encoder = CLIPTextModel.from_pretrained(
    SD15_MODEL_ID, subfolder='text_encoder', torch_dtype=torch.float16).to(device)
text_encoder.requires_grad_(False)
text_encoder.eval()

# SD 1.5 UNet — frozen backbone; receives ControlNet residuals
unet = UNet2DConditionModel.from_pretrained(
    SD15_MODEL_ID, subfolder='unet', torch_dtype=torch.float16).to(device)
unet.requires_grad_(False)
unet.enable_gradient_checkpointing()  # saves ~400MB activation memory

# DDPM training scheduler
noise_scheduler = DDPMScheduler.from_pretrained(SD15_MODEL_ID, subfolder='scheduler')

# ControlNet — init from SD 1.5 UNet encoder (standard practice)
if RESUME_CHECKPOINT and os.path.exists(RESUME_CHECKPOINT):
    controlnet = ControlNetModel.from_pretrained(
        RESUME_CHECKPOINT, torch_dtype=torch.float32).to(device)
    print(f'Resumed ControlNet from: {RESUME_CHECKPOINT}')
else:
    # Standard ControlNet init: copy encoder weights from UNet, zero-init output convs
    controlnet = ControlNetModel.from_unet(unet).to(device)
    controlnet = controlnet.to(torch.float32)  # train ControlNet in fp32 for stability
    print('ControlNet initialized from SD 1.5 UNet encoder weights.')

controlnet.enable_gradient_checkpointing()

# Pre-compute text embeddings for the fixed prompt (done once, reused every batch)
with torch.no_grad():
    tok = tokenizer(
        [PROMPT],
        padding='max_length',
        max_length=tokenizer.model_max_length,
        truncation=True,
        return_tensors='pt',
    )
    text_embeds = text_encoder(tok.input_ids.to(device))[0]  # [1, 77, 768] fp16

    # Unconditional embeddings for classifier-free guidance (used at inference)
    tok_uncond = tokenizer(
        [''],
        padding='max_length',
        max_length=tokenizer.model_max_length,
        truncation=True,
        return_tensors='pt',
    )
    uncond_embeds = text_encoder(tok_uncond.input_ids.to(device))[0]  # [1, 77, 768] fp16

print(f'\nText embeddings shape: {text_embeds.shape}')

# Model parameter counts
n_cn  = sum(p.numel() for p in controlnet.parameters() if p.requires_grad)
n_tot = sum(p.numel() for p in list(vae.parameters()) +
            list(text_encoder.parameters()) + list(unet.parameters()))
print(f'\nControlNet trainable params: {n_cn:,}  (~{n_cn/1e6:.0f}M)')
print(f'Frozen SD 1.5 params:        {n_tot:,}  (~{n_tot/1e6:.0f}M)')

if device.type == 'cuda':
    print(f'\nGPU memory after model load: '
          f'{torch.cuda.memory_allocated() / 1e9:.2f} GB '
          f'/ {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

In [ ]:
# 7. Optimizer + LR scheduler

optimizer = optim.AdamW(
    controlnet.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)

# Linear warmup → cosine decay
total_steps = NUM_EPOCHS * len(train_loader)
def lr_lambda(step):
    if step < LR_WARMUP_STEPS:
        return float(step) / max(1, LR_WARMUP_STEPS)
    progress = float(step - LR_WARMUP_STEPS) / max(1, total_steps - LR_WARMUP_STEPS)
    return max(0.0, 0.5 * (1.0 + math.cos(math.pi * progress)))
lr_scheduler = optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)

# AMP scaler for ControlNet (fp32) forward pass mixed with fp16 frozen models
scaler = GradScaler('cuda')

START_EPOCH  = 1
global_step  = 0
history      = {'loss': [], 'lr': [], 'epoch_time': []}

# VAE scale factor (SD 1.5 standard: 0.18215)
VAE_SCALE = vae.config.scaling_factor
print(f'VAE scaling factor: {VAE_SCALE}')
print(f'Optimizer: AdamW  lr={LEARNING_RATE}  warmup={LR_WARMUP_STEPS} steps')
print(f'Total training steps: {total_steps}')

In [ ]:
# 8. Training loop — ControlNet ε-prediction in latent space
#
# Per-step:
#   1. Encode clean image x0 → VAE latent z0  (frozen VAE, fp16)
#   2. Sample random timestep t, add noise → z_t
#   3. ControlNet(z_t, t, text_embeds, hint) → residual features  (fp32 ControlNet)
#   4. UNet(z_t, t, text_embeds, residuals) → predicted noise ε   (frozen UNet, fp16)
#   5. MSE(ε_pred, ε_true)  ← loss in latent space

def encode_images(images):
    """Encode [B,3,H,W] images in [-1,1] to VAE latents [B,4,h,w] scaled by VAE_SCALE."""
    with torch.no_grad():
        latents = vae.encode(images.to(dtype=torch.float16)).latent_dist.sample()
    return latents * VAE_SCALE


def save_controlnet_samples(controlnet, unet, vae, test_dataset, device, epoch,
                            sample_dir, n_samples=3):
    """Generate samples via DDIM from test hints, save visual grid."""
    controlnet.eval()
    ddim_sched = DDIMScheduler.from_pretrained(SD15_MODEL_ID, subfolder='scheduler')
    ddim_sched.set_timesteps(DDIM_STEPS)

    n   = min(n_samples, len(test_dataset))
    fig, axs = plt.subplots(n, 3, figsize=(15, 5 * n))
    if n == 1:
        axs = axs[np.newaxis, :]

    for j, ttl in enumerate(['Binary Leaf Mask', 'Generated (DDIM)', 'Real GLS (ref)']):
        axs[0, j].set_title(ttl, fontsize=11, fontweight='bold')

    def t2np(t):
        return ((t.clamp(-1, 1) + 1) / 2).cpu().permute(1, 2, 0).float().numpy()

    with torch.no_grad():
        # Batch text embeds for CFG: [uncond, cond]
        cfg_embeds = torch.cat([uncond_embeds, text_embeds], dim=0)  # [2, 77, 768]

        for i in range(n):
            real_img, hint_t, fname = test_dataset[i]
            hint_in = hint_t.unsqueeze(0).to(device, dtype=torch.float32)

            # Start from Gaussian noise in latent space
            lat = torch.randn(1, 4, IMAGE_SIZE // 8, IMAGE_SIZE // 8,
                              device=device, dtype=torch.float16)

            for t_val in ddim_sched.timesteps:
                lat_in  = torch.cat([lat, lat], dim=0)       # duplicate for CFG
                t_batch = t_val.unsqueeze(0).repeat(2).to(device)

                # ControlNet residuals
                hint_cfg = hint_in.repeat(2, 1, 1, 1).to(dtype=torch.float32)
                cn_out   = controlnet(
                    lat_in.float(), t_batch,
                    encoder_hidden_states=cfg_embeds,
                    controlnet_cond=hint_cfg,
                    return_dict=False,
                )
                down_res, mid_res = cn_out[:-1], cn_out[-1]

                # UNet noise prediction
                noise_pred = unet(
                    lat_in, t_batch,
                    encoder_hidden_states=cfg_embeds,
                    down_block_additional_residuals=[
                        r.to(dtype=torch.float16) for r in down_res],
                    mid_block_additional_residual=mid_res.to(dtype=torch.float16),
                ).sample

                # CFG: uncond + guidance * (cond - uncond)
                eps_uncond, eps_cond = noise_pred.chunk(2)
                eps = eps_uncond + GUIDANCE_SCALE * (eps_cond - eps_uncond)

                lat = ddim_sched.step(eps, t_val, lat).prev_sample

            # Decode latent → pixel image
            decoded = vae.decode(lat.to(dtype=torch.float16) / VAE_SCALE).sample

            mask_vis = hint_t[0].cpu().numpy()  # single channel of hint
            axs[i, 0].imshow(mask_vis, cmap='gray'); axs[i, 0].axis('off')
            axs[i, 0].set_ylabel(fname[:20], fontsize=7)
            axs[i, 1].imshow(t2np(decoded.squeeze(0))); axs[i, 1].axis('off')
            axs[i, 2].imshow(t2np(real_img)); axs[i, 2].axis('off')

    plt.suptitle(f'SD 1.5 ControlNet Samples — Epoch {epoch}',
                 fontsize=14, fontweight='bold')
    plt.tight_layout()
    path = os.path.join(sample_dir, f'samples_epoch{epoch:04d}.png')
    plt.savefig(path, dpi=120, bbox_inches='tight')
    plt.close(fig)
    controlnet.train()


# --- Training ---
print(f'Training from epoch {START_EPOCH} to {NUM_EPOCHS}')
print('=' * 80)

for epoch in range(START_EPOCH, NUM_EPOCHS + 1):
    epoch_start = time.time()
    controlnet.train()

    run_loss  = 0.0
    n_samples = 0
    optimizer.zero_grad()

    progress = tqdm(train_loader, desc=f'Epoch {epoch}/{NUM_EPOCHS}')
    for step, (clean_img, hint, _) in enumerate(progress):
        B         = clean_img.size(0)
        clean_img = clean_img.to(device)               # [B,3,H,W] in [-1,1]
        hint      = hint.to(device, dtype=torch.float32)  # [B,3,H,W] in {0,1}

        # Encode to latent space (frozen VAE, no grad)
        latents = encode_images(clean_img)  # [B, 4, 64, 64]

        # Sample noise + timestep
        noise = torch.randn_like(latents)
        t     = torch.randint(0, noise_scheduler.config.num_train_timesteps,
                              (B,), device=device, dtype=torch.long)

        # Add noise to latents (forward diffusion)
        z_t = noise_scheduler.add_noise(latents, noise, t)

        # Expand text embeds to batch
        enc_hid = text_embeds.repeat(B, 1, 1).to(dtype=torch.float16)  # [B, 77, 768]

        with autocast('cuda'):
            # ControlNet forward (fp32 ControlNet + fp16 context via autocast)
            cn_result = controlnet(
                z_t.float(), t,
                encoder_hidden_states=enc_hid,
                controlnet_cond=hint,
                return_dict=False,
            )
            down_residuals = cn_result[:-1]
            mid_residual   = cn_result[-1]

            # Frozen UNet noise prediction
            noise_pred = unet(
                z_t, t,
                encoder_hidden_states=enc_hid,
                down_block_additional_residuals=[
                    r.to(dtype=torch.float16) for r in down_residuals],
                mid_block_additional_residual=mid_residual.to(dtype=torch.float16),
            ).sample

            loss = F.mse_loss(noise_pred.float(), noise.float()) / GRAD_ACCUM

        scaler.scale(loss).backward()

        if (step + 1) % GRAD_ACCUM == 0 or (step + 1) == len(train_loader):
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(controlnet.parameters(), max_norm=1.0)
            scaler.step(optimizer)
            scaler.update()
            lr_scheduler.step()
            optimizer.zero_grad()
            global_step += 1

        run_loss  += loss.item() * GRAD_ACCUM * B
        n_samples += B
        progress.set_postfix({'MSE': f'{loss.item() * GRAD_ACCUM:.5f}'})

    avg_loss   = run_loss / n_samples
    epoch_time = time.time() - epoch_start
    lr_now     = lr_scheduler.get_last_lr()[0]

    history['loss'].append(avg_loss)
    history['lr'].append(lr_now)
    history['epoch_time'].append(epoch_time)

    print(f'Epoch {epoch}/{NUM_EPOCHS} [{epoch_time:.1f}s | lr={lr_now:.2e}] | '
          f'MSE loss: {avg_loss:.6f}')

    # Save checkpoint (ControlNet weights only)
    if epoch % SAVE_EVERY == 0:
        ckpt_dir = os.path.join(CHECKPOINT_DIR, f'controlnet_epoch{epoch}')
        controlnet.save_pretrained(ckpt_dir)
        ckpt_meta = {
            'epoch': epoch, 'loss': avg_loss,
            'global_step': global_step, 'history': history,
        }
        with open(os.path.join(ckpt_dir, 'meta.json'), 'w') as f:
            json.dump(ckpt_meta, f, indent=2)
        print(f'  >> ControlNet saved: {ckpt_dir}')

    # Save visual samples
    if epoch % SAMPLE_EVERY == 0 or epoch == 1:
        save_controlnet_samples(controlnet, unet, vae, test_ds, device,
                                epoch, SAMPLE_DIR)
        print(f'  >> Samples saved to {SAMPLE_DIR}')

    torch.cuda.empty_cache()

# Final save
final_dir = os.path.join(CHECKPOINT_DIR, f'controlnet_epoch{NUM_EPOCHS}_final')
controlnet.save_pretrained(final_dir)
with open(os.path.join(final_dir, 'meta.json'), 'w') as f:
    json.dump({'epoch': NUM_EPOCHS, 'loss': history['loss'][-1],
               'global_step': global_step, 'history': history}, f, indent=2)
print(f'Final ControlNet saved: {final_dir}')

log_path = os.path.join(LOG_DIR, 'sd15_controlnet_history.json')
with open(log_path, 'w') as f:
    json.dump(history, f, indent=2)
print(f'Training history saved: {log_path}')

In [ ]:
# 9. Training metrics plots

epochs = range(1, len(history['loss']) + 1)
fig, axs = plt.subplots(1, 2, figsize=(14, 5))

axs[0].plot(epochs, history['loss'], 'b-', linewidth=1.5)
axs[0].set_title('ControlNet MSE Loss (ε-prediction, latent space)')
axs[0].set_xlabel('Epoch'); axs[0].set_ylabel('MSE'); axs[0].grid(True)
axs[0].set_yscale('log')

axs[1].plot(epochs, history['lr'], 'g-', linewidth=1.5)
axs[1].set_title('Learning Rate (warmup + cosine decay)')
axs[1].set_xlabel('Epoch'); axs[1].set_ylabel('LR'); axs[1].grid(True)

plt.suptitle('SD 1.5 ControlNet Training Metrics', fontsize=14, fontweight='bold')
plt.tight_layout()
fig_path = os.path.join(LOG_DIR, 'sd15_controlnet_loss_curves.png')
plt.savefig(fig_path, dpi=150, bbox_inches='tight')
plt.show()
print(f'Loss curves saved: {fig_path}')

total_time = sum(history['epoch_time'])
print(f'\nFinal MSE loss:      {history["loss"][-1]:.6f}')
print(f'Total training time: {total_time / 3600:.2f} hours')

## Inference

Cells below use the trained ControlNet with DDIM sampling and classifier-free guidance.
Load a saved checkpoint before running.

In [ ]:
# 10. Load checkpoint for inference
INFER_CHECKPOINT = os.path.join(CHECKPOINT_DIR,
                                f'controlnet_epoch{NUM_EPOCHS}_final')

controlnet_infer = ControlNetModel.from_pretrained(
    INFER_CHECKPOINT, torch_dtype=torch.float16).to(device)
controlnet_infer.eval()

with open(os.path.join(INFER_CHECKPOINT, 'meta.json')) as f:
    meta = json.load(f)
print(f'Loaded checkpoint from epoch {meta["epoch"]}  (loss: {meta["loss"]:.6f})')
print(f'\nTest set: {len(test_ds)} images')
for i in range(len(test_ds)):
    _, _, fname = test_ds[i]
    print(f'  [{i:2d}] {fname}')

In [ ]:
# 11. Single test sample inference
TEST_INDEX     = 0       # <-- change this
INFER_GUIDANCE = 7.5     # classifier-free guidance scale
INFER_CN_SCALE = 1.0     # ControlNet conditioning scale

torch.cuda.empty_cache()

ddim_sched = DDIMScheduler.from_pretrained(SD15_MODEL_ID, subfolder='scheduler')
ddim_sched.set_timesteps(DDIM_STEPS)

real_img, hint_t, fname = test_ds[TEST_INDEX]
hint_in = hint_t.unsqueeze(0).to(device, dtype=torch.float16)

def generate_from_hint(hint, guidance_scale=7.5, seed=None):
    if seed is not None:
        torch.manual_seed(seed)
    lat = torch.randn(1, 4, IMAGE_SIZE // 8, IMAGE_SIZE // 8,
                      device=device, dtype=torch.float16)
    cfg_embeds = torch.cat([uncond_embeds, text_embeds], dim=0)

    with torch.no_grad():
        for t_val in ddim_sched.timesteps:
            lat_in   = torch.cat([lat, lat], dim=0)
            t_batch  = t_val.unsqueeze(0).repeat(2).to(device)
            hint_cfg = hint.repeat(2, 1, 1, 1)

            cn_out = controlnet_infer(
                lat_in, t_batch,
                encoder_hidden_states=cfg_embeds,
                controlnet_cond=hint_cfg,
                conditioning_scale=INFER_CN_SCALE,
                return_dict=False,
            )
            down_res, mid_res = cn_out[:-1], cn_out[-1]

            noise_pred = unet(
                lat_in, t_batch,
                encoder_hidden_states=cfg_embeds,
                down_block_additional_residuals=down_res,
                mid_block_additional_residual=mid_res,
            ).sample

            eps_u, eps_c = noise_pred.chunk(2)
            eps = eps_u + guidance_scale * (eps_c - eps_u)
            lat = ddim_sched.step(eps, t_val, lat).prev_sample

        decoded = vae.decode(lat / VAE_SCALE).sample
    return decoded.squeeze(0)


def t2np(t):
    return ((t.clamp(-1, 1) + 1) / 2).cpu().float().permute(1, 2, 0).numpy()


gen_det  = generate_from_hint(hint_in, guidance_scale=7.5, seed=0)
gen_stoc = generate_from_hint(hint_in, guidance_scale=7.5, seed=None)

fig, axs = plt.subplots(1, 4, figsize=(24, 5))
axs[0].imshow(hint_t[0].cpu().numpy(), cmap='gray')
axs[0].set_title(f'Input Mask\n{fname}'); axs[0].axis('off')
axs[1].imshow(t2np(gen_det))
axs[1].set_title(f'Generated (seed=0)\nDDIM {DDIM_STEPS} steps, cfg={INFER_GUIDANCE}')
axs[1].axis('off')
axs[2].imshow(t2np(gen_stoc))
axs[2].set_title(f'Generated (random seed)\nDDIM {DDIM_STEPS} steps'); axs[2].axis('off')
axs[3].imshow(t2np(real_img))
axs[3].set_title('Real GLS Image (reference)'); axs[3].axis('off')

plt.tight_layout()
plt.savefig(os.path.join(SAMPLE_DIR, f'infer_test{TEST_INDEX}.png'),
            dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# 12. Diversity evaluation — N seeds per test mask
# Mirrors diversity_eval in diffusion_train_linux.ipynb for direct comparison.
# High mean pairwise L1 > 0.05 = model is not collapsed.

torch.cuda.empty_cache()

N_SEEDS = 5
N_EVAL  = min(10, len(test_ds))

print(f'Evaluating {N_EVAL} test masks × {N_SEEDS} seeds each ...')
diversity_l1 = []

fig, axs = plt.subplots(N_EVAL, N_SEEDS + 1, figsize=(4 * (N_SEEDS + 1), 4 * N_EVAL))
if N_EVAL == 1:
    axs = axs[np.newaxis, :]

for i in range(N_EVAL):
    _, hint_t, fname = test_ds[i]
    hint_in = hint_t.unsqueeze(0).to(device, dtype=torch.float16)
    samples = []

    for s in range(N_SEEDS):
        gen = generate_from_hint(hint_in, guidance_scale=INFER_GUIDANCE, seed=None)
        samples.append(gen.cpu())

    pairs = []
    for a in range(N_SEEDS):
        for b in range(a + 1, N_SEEDS):
            pairs.append(torch.mean(torch.abs(samples[a] - samples[b])).item())
    diversity_l1.append(float(np.mean(pairs)))

    axs[i, 0].imshow(hint_t[0].numpy(), cmap='gray')
    axs[i, 0].set_title(f'Mask\n{fname[:15]}', fontsize=7); axs[i, 0].axis('off')
    for s, samp in enumerate(samples):
        axs[i, s + 1].imshow(t2np(samp))
        axs[i, s + 1].set_title(f'Seed {s}', fontsize=7); axs[i, s + 1].axis('off')

plt.suptitle(
    f'SD 1.5 ControlNet Diversity  |  Mean pairwise L1: {np.mean(diversity_l1):.4f}',
    fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(SAMPLE_DIR, 'eval_diversity.png'), dpi=100, bbox_inches='tight')
plt.show()

print('=' * 55)
print('DIVERSITY EVALUATION (SD 1.5 ControlNet)')
print('=' * 55)
for i, d in enumerate(diversity_l1):
    _, _, fname = test_ds[i]
    print(f'  [{i:2d}] {fname[:25]:25s}: mean pairwise L1 = {d:.4f}')
print(f'  Overall mean L1: {np.mean(diversity_l1):.4f}  (>0.05 = not collapsed)')
print('=' * 55)

In [ ]:
# 13. Inference on a custom binary mask
# Change CUSTOM_MASK_PATH to any binary PNG (white = leaf, black = background).

CUSTOM_MASK_PATH = os.path.join(
    str(repo_root), 'data', 'masks', 'leaf', 'gls', 'leaf_mask_001.png')  # <-- change this

torch.cuda.empty_cache()

mask_img = Image.open(CUSTOM_MASK_PATH).convert('L')
mask_img = mask_img.resize((IMAGE_SIZE, IMAGE_SIZE), Image.NEAREST)
mask_t   = (TF.to_tensor(mask_img) > 0.5).float()          # [1, H, W]
hint_t   = mask_t.repeat(3, 1, 1).unsqueeze(0).to(device, dtype=torch.float16)  # [1, 3, H, W]

gen_det  = generate_from_hint(hint_t, guidance_scale=7.5, seed=0)
gen_stoc = generate_from_hint(hint_t, guidance_scale=7.5, seed=None)

fig, axs = plt.subplots(1, 3, figsize=(18, 5))
axs[0].imshow(mask_t.squeeze(0).numpy(), cmap='gray')
axs[0].set_title(f'Input Mask\n{os.path.basename(CUSTOM_MASK_PATH)}')
axs[0].axis('off')
axs[1].imshow(t2np(gen_det))
axs[1].set_title('Generated — seed=0'); axs[1].axis('off')
axs[2].imshow(t2np(gen_stoc))
axs[2].set_title('Generated — random seed'); axs[2].axis('off')

plt.tight_layout()
plt.savefig(os.path.join(SAMPLE_DIR, 'inference_custom.png'), dpi=150, bbox_inches='tight')
plt.show()